# Motion-Corrected Image Reconstruction (MCIR)


In [ ]:
__version__ = '0.1.0'

# import engine module
import sirf.Gadgetron as pMR
from sirf.Utilities import examples_data_path
# from sirf_exercises import exercises_data_pathtoken
import sirf.Reg as pReg


from cil.framework import  AcquisitionGeometry, BlockDataContainer, BlockGeometry
from cil.optimisation.functions import Function, OperatorCompositionFunction, SmoothMixedL21Norm, L1Norm, L2NormSquared, BlockFunction, MixedL21Norm, IndicatorBox, TotalVariation, LeastSquares, ZeroFunction
from cil.optimisation.operators import GradientOperator, BlockOperator, ZeroOperator, CompositionOperator,LinearOperator
from cil.optimisation.algorithms import PDHG, FISTA, GD, SPDHG
from cil.plugins.ccpi_regularisation.functions import FGP_TV

# import further modules
import os
import numpy as np
import scipy.signal as sp_signal

import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

import functools

In [ ]:
# Plotting function
data_path = '/home/jovyan/devel/SIRF-Contribs/src/notebooks/'
os.chdir(data_path)
import utils
from utils import plot_rpe_3d, plot_rpe_3d_simple
print ("Ciao")


In [ ]:
import importlib
importlib.reload(utils)
from utils import plot_rpe_3d, plot_rpe_3d_simple, saveCallback, add_HORIZONTAL_phase_shift

In [ ]:
############# SETUP #################
# number of motion states
Nms = 4
# max
max_shift = 20.
# setup linear shifts
shifts = np.linspace(0, 1, Nms) * max_shift

## Recon parameters
n_epochs = 100
# factor in front of L2Norm
omega = 0.5
# regularisation parameter
alpha = 2e-5


In [ ]:
# data_path = exercises_data_path('MR')

filename = os.path.join(data_path, '3D_GRPE_no_motion.h5')

In [ ]:
# Load in the data
acq_data = pMR.AcquisitionData(filename)
acq_data.sort_by_time()

In [ ]:
csm = pMR.CoilSensitivityData()
csm.smoothness = 100
csm.calculate(acq_data) 

E = pMR.AcquisitionModel(acqs=acq_data, imgs=csm)
E.set_coil_sensitivity_maps(csm)


# Adding horizontal phase shift

## Create Motion States

In [ ]:
import importlib
importlib.reload(utils)

from utils import split_data_with_horizontal_shifts, create_AcquisitionModel_for_each_motion_state 

acq_ms = split_data_with_horizontal_shifts(acq_data, shifts)

E_ms = create_AcquisitionModel_for_each_motion_state(acq_ms, csm)




In [ ]:
import functools
im_fista_ms = [0] * Nms

im_ms = E_ms[0].inverse(acq_ms[0])

for ind in range(Nms):

    # Starting image
    x_init = im_ms.clone()
    x_init.fill(0.0)

    # Objective function
    f = LeastSquares(E_ms[ind], acq_ms[ind], c=1)
    G = ZeroFunction()

    # Set up FISTA for least squares
    fista = FISTA(initial=x_init, f=f, g=G)
    fista.max_iteration = 100
    fista.update_objective_interval = 5

    # Run FISTA
    sc = functools.partial(saveCallback, fista, f"FISTA_L2_ms_{ind}", os.path.join(data_path, 'recons', f'MS_{Nms}'))
    fista.run(10, verbose=1, callback=sc)
    
    # Get result
    im_fista_ms[ind] = fista.get_output()

In [ ]:
# Prepare results
#%matplotlib inline
im_fista_ms_arr = []
im_fista_diff_arr = []
for ind in range(Nms):
    im_fista_ms_arr.append(im_fista_ms[ind].as_array())
    im_fista_diff_arr.append(np.abs(im_fista_ms[ind].as_array()) - np.abs(im_fista_ms_arr[0]))
    
# Visualise different motion states
plot_rpe_3d_simple(im_fista_ms_arr[:], [64, 64], [ f'MS {ms}' for ms in range(Nms) ])
print('\n')
# Visualise difference to first motion state
plot_rpe_3d_simple(im_fista_diff_arr[:], [64, 64], [f'MS {ms}- MS 0'  for ms in range(Nms) ])

## Create Motion Vector fields from known shifts

In [ ]:
# Motion fransformation object
mf_resampler = [0] * Nms

# Forward transformation (i.e. reference image transformed to current motion state)
im_forward = [0] * Nms

# Backward transformation (i.e. current motion image transformed to reference motion state)
im_backward = [0] * Nms

tms = []
import logging

for shift in shifts:
    # fix the pixel spacing
    spacing = im_fista_ms[0].get_geometrical_info().get_spacing()
    # There is inconsistency in SIRF wrt to the order of the dimensions
    # here we rely on the fact that the spacing is uniform.
    uniform = (spacing[0] == spacing[1]) and (spacing[1] == spacing[2]) and (spacing[0] == spacing[2])
    if not uniform:
        logging.warn('The spacing is not uniform. The results might be incorrect.')
    t_x=shift * spacing[0]
    t_y=0
    tm = pReg.AffineTransformation(np.array(\
            [[1, 0, 0, t_x], \
             [0, 1, 0, t_y], \
             [0, 0, 1, 0  ], \
             [0, 0, 0, 1  ]]))
    tms.append(tm)
    
for ind in range(Nms):

    # Create resampler
    mf_resampler[ind] = pReg.NiftyResample()
    # check units used by the resampler
    mf_resampler[ind].set_reference_image(im_fista_ms[0])
    mf_resampler[ind].set_floating_image(im_fista_ms[ind])
    mf_resampler[ind].add_transformation(tms[ind])
    mf_resampler[ind].set_padding_value(0)
    mf_resampler[ind].set_interpolation_type_to_linear()

    im_forward[ind] = mf_resampler[ind].forward(im_fista_ms[0])
    im_backward[ind] = mf_resampler[ind].backward(im_fista_ms[ind])

## Let's look at the backward transformed images (i.e. the images transformed to the reference motion state).

In [ ]:
# Prepare results
%matplotlib inline
im_backward_arr = []
im_backward_diff_arr = []
for ind in range(Nms):
    im_backward_arr.append(im_backward[ind].as_array())
    im_backward_diff_arr.append(np.abs(im_backward[ind].as_array()) - np.abs(im_backward[0].as_array()))
    
# Visualise different motion states transformed to reference motion state
plot_rpe_3d_simple(im_backward_arr, [64, 64], [ f'MS {ms}' for ms in range(Nms) ])

# Visualise difference to first motion state
plot_rpe_3d_simple(im_backward_diff_arr, [64, 64], [f'MS {ms}- MS 0'  for ms in range(Nms) ])


## Modifications to CIL/SIRF (check comments to see whether cell(s) can be removed)

In [ ]:
from pReg import NiftyResampler 
from cil.optimisation.operators import LinearOperator


In [ ]:
# this cell is to be removed if using newer CIL version that implements 1470 (leave default, symmetric=False)

# Overriding PowerMethod to ensure no bug
class MyLinearOperator(LinearOperator):
    
    def PowerMethod(operator, max_iteration=10, initial=None, tolerance = 1e-5,  return_all=False):

        symmetric = False
        try:
            if operator.domain_geometry()==operator.range_geometry():
                symmetric = True
        except AssertionError:
            pass

        if initial is None:
            x0 = operator.domain_geometry().allocate('random')
        else:
            x0 = initial.copy()

        y_tmp = operator.range_geometry().allocate()
  
        # Normalize first eigenvector
        x0_norm = x0.norm()
        x0 /= x0_norm
        
        # initial guess for dominant eigenvalue
        eig_old = 1.
        eig_list = []
        diff = np.finfo('d').max
        
        i=0
        while (i < max_iteration and diff > tolerance):
            i+=1
            operator.direct(x0, out = y_tmp)            
            operator.adjoint(y_tmp,out=x0)
            
            # Get eigenvalue using Rayleigh quotient: denominator=1, due to normalization
            x0_norm = x0.norm()      
            x0 /= x0_norm

            eig_new =  np.abs(x0_norm)
            if not symmetric:
                eig_new = np.sqrt(eig_new)

            eig_list.append(eig_new)
            eig_old = eig_new      
        
        if return_all:
            return eig_new, i, x0, eig_list
        else:
            return eig_new
        
    def calculate_norm(self):
        
        r""" Returns the norm of the LinearOperator calculated by the PowerMethod with default values.
                """
        return MyLinearOperator.PowerMethod(self)            


In [ ]:
# this cell is to be removed if using newer SIRF version that implements 1182

def NiftyResampler_norm(x):
    out = MyLinearOperator.PowerMethod(x)
    return out

setattr(NiftyResampler, 'norm', NiftyResampler_norm)

In [ ]:
# this cell is to be removed if using newer CIL version that implements 1472

def CompositionOperator_norm(comp_operator):
    out = 1.
    for factor in comp_operator.operators:
        out = out * factor.norm()
    return out

setattr(CompositionOperator, 'norm', CompositionOperator_norm)

In [ ]:
# this cell is to be removed if using newer CIL version that implements 1479


import types   # import needed for later

# Overriding proximal, to remove call to function "check_input"
def new_proximal(self, x, tau, out=None):
    arr = x.as_array()
    if np.iscomplexobj(arr):
        # do real and imag part indep
        in_arr = np.asarray(arr.real, dtype=np.float32, order='C')
        res, info = self.proximal_numpy(in_arr, tau)
        arr.real = res[:]
        in_arr = np.asarray(arr.imag, dtype=np.float32, order='C')
        res, info = self.proximal_numpy(in_arr, tau)
        arr.imag = res[:]
        self.info = info
        if out is not None:
            out.fill(arr)
        else:
            out = x.copy()
            out.fill(arr)
            return out
    else:
        arr = np.asarray(x.as_array(), dtype=np.float32, order='C')
        res, info = self.proximal_numpy(arr, tau)
        self.info = info
        if out is not None:
            out.fill(res)
        else:
            out = x.copy()
            out.fill(res)
            return out

## MCIR with PDHG

In [ ]:
from datetime import datetime

In [ ]:
C = [CompositionOperator(am, res) for am, res in zip(*(E_ms, mf_resampler))]
A = BlockOperator(*C)

acq_ms_block = BlockDataContainer(*acq_ms)
# x_init = A.adjoint(acq_ms_block)
# x_init.fill(0.0)

F_all_subs = []
for i in acq_ms:
    F_all_subs.append(omega*L2NormSquared(b=i))
    
F = BlockFunction(*F_all_subs)

In [ ]:
G = alpha * FGP_TV(device='cpu', nonnegativity=False)

# this line is to be removed if using newer CIL version that implements 1479
G.proximal = types.MethodType(new_proximal, G)


pdhg_2 = PDHG(
            f=F, g=G, operator=A,
            max_iteration=n_epochs,
            update_objective_interval=10)


In [ ]:
sc = functools.partial(saveCallback, pdhg_2, f"PDHG_TV_{Nms:02d}ms", os.path.join(data_path, 'recons', f'MS_{Nms}'))
    
start = datetime.now()
pdhg_2.run(verbose=2, callback=sc)
end = datetime.now()

pdhg_2_time = end-start

In [ ]:
im = pdhg_2.get_output().as_array()

%matplotlib inline
plot_rpe_3d([im],
            [59, 64],
            ['PDHG, 4ms'],
            [3*10**(-5)],
            'PDHG, 4ms.png')

## MCIR with SPDHG

In [ ]:
# reuse the same TV instance
# G = alpha * FGP_TV(device='cpu', nonnegativity=False)
# G.proximal = types.MethodType(new_proximal, G)

start = datetime.now()
spdhg_2 = SPDHG(
            f = F, g = G, operator = A, 
            max_iteration = n_epochs*Nms, 
            update_objective_interval = 10*Nms)
  

In [ ]:
sc = functools.partial(saveCallback, pdhg_2, f"SPDHG_TV_{Nms:02d}ms", os.path.join(data_path, 'recons', f'MS_{Nms}'))
    
start = datetime.now()
spdhg_2.run(verbose=2, callback=sc)
end = datetime.now()

spdhg_2_time = end-start

In [ ]:
im = spdhg_2.get_output().as_array()

%matplotlib inline
plot_rpe_3d([im],
            [59, 64],
            [f'SPDHG, {Nms}ms'],
            [3*10**(-5)],
            f'SPDHG, {Nms}ms.png')

In [ ]:
pdhg_2_time.seconds/3600

In [ ]:
spdhg_2_time.seconds/3600

In [ ]:
np.save(os.path.join(data_path, 'recons', f'MS_{Nms}', 'PDHG_objective.npy'), np.asarray(pdhg_2.objective))

In [ ]:
np.save(os.path.join(data_path, 'recons', f'MS_{Nms}', 'SPDHG_objective.npy'), np.asarray(spdhg_2.objective))